In [75]:
1

1

In [76]:
1

1

In [77]:
import os
import subprocess

def check_docker_image(image_name):
    """Check if the specified Docker image is available locally."""
    try:
        result = subprocess.run(['docker', 'images', '--format', '{{.Repository}}:{{.Tag}}'], capture_output=True, text=True)
        available_images = result.stdout.splitlines()
        return image_name in available_images
    except Exception as e:
        print(f"Error checking Docker image: {e}")
        return False

def pull_docker_image(image_name):
    """Pull the specified Docker image."""
    try:
        subprocess.run(['docker', 'pull', image_name], capture_output=True, text=True)
        print(f"Pulled Docker image: {image_name}")
    except Exception as e:
        print(f"Error pulling Docker image: {e}")

def start_docker_container(image_name, port_mapping, volume_mapping):
    """Start the Docker container."""
    try:
        subprocess.run(
            ['docker', 'run', '-d', '-p', port_mapping, '-v', volume_mapping, image_name],
            capture_output=True, text=True
        )
        print(f"Started Docker container with image: {image_name}")
    except Exception as e:
        print(f"Error starting Docker container: {e}")

def ensure_docker_container(image_name, port_mapping, volume_mapping):
    """Ensure the Docker image is available and a container is running."""
    if not check_docker_image(image_name):
        print(f"Image {image_name} not found locally. Pulling it now...")
        pull_docker_image(image_name)
    else:
        print(f"Image {image_name} is already available locally.")
    
    start_docker_container(image_name, port_mapping, volume_mapping)




# Configuration
image_name = "qdrant/qdrant:latest"  # Replace with your Qdrant Docker image name
port_mapping = "6333:6333"  # Replace with your port mapping
volume_mapping = f'{os.getcwd()}/qdrant_db_ad:/qdrant/storage'  # Replace with your volume mapping
qdrant_url = "http://localhost:6333"  # Replace with your Qdrant URL and port
# # Ensure the Docker container is running
ensure_docker_container(image_name, port_mapping, volume_mapping)

1

In [65]:
from qdrant_client import QdrantClient
from qdrant_client.http import models 
from qdrant_client.http.models import CollectionStatus

/Users/s748779/IAG_test/test1/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [78]:
client = QdrantClient(host="localhost", port = 6333)

In [67]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/s748779/IAG_test/test1/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/Users/s748779/IAG_test/test1/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [73]:
# match it only with problem tags 

def top_n_ads(n, input_content_to_match, tags=[], collection_name):
    '''
    n : top n matches 
    tags : problem, solution, options, context
    collection_name : collection to be used 
    '''
    
    
    if tags:
        hits = client.search(
        collection_name=collection_name,
        query_vector=embedding_model.encode(input_content_to_match).tolist(),
        query_filter=models.Filter(**{
                                    "must": [
                                              {
                                                "key": "tag",
                                                "match": {
                                                  "any": tags
                                                }
                                              }
                                            ]        
                                        }),
        limit=n,
        )
        return hits
    
    hits = client.search(
        collection_name="rnw_adr_collection_3",
        query_vector=embedding_model.encode(input_content_to_match).tolist(),
        limit = n
     )
    
    return hits



In [10]:
hits = client.search(
    collection_name="rnw_adr_collection",
    query_vector=embedding_model.encode("payments").tolist(),
    limit=15,
)
for hit in hits:
    print(hit.payload, "score:", hit.score)

{'ad_ref': 'AD 198 - Which payment vendor/s and redundancy pattern will provide resilient online payments for West 2?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590408440', 'page_ref': '590408440', 'tag': 'solution'} score: 0.48636675
{'ad_ref': 'AD 129 - Where are Finance Provider Entities mastered?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590233697', 'page_ref': '590233697', 'tag': 'solution'} score: 0.45952898
{'ad_ref': 'AD 137 - Which system will provide API access to the payment provider for CGU Direct Home, Motor & Niche?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590265508', 'page_ref': '590265508', 'tag': 'solution'} score: 0.37222672
{'ad_ref': 'AD 054 - Which vendor will provide real-time customer payment capabilities?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/S

In [118]:
hits = client.search(
    collection_name="rnw_adr_collection",
    query_vector=embedding_model.encode("Which adr tells me how to make a web based decision?").tolist(),
    ),
    limit=15,
)
for hit in hits:
    print(hit.payload, "score:", hit.score)

{'ad_ref': "AD 169 - Which system/s will provide API Gateway for IAG's Partners?", 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590348418', 'page_ref': '590348418', 'tag': 'solution'} score: 0.4572095
{'ad_ref': 'AD 076 - Which system/s will provide website content management?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590119191', 'page_ref': '590119191', 'tag': 'solution'} score: 0.4296944
{'ad_ref': 'AD 030 - Which technology will used to build exposure APIs in PC/BC?', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590095558', 'page_ref': '590095558', 'tag': 'solution'} score: 0.42023975
{'ad_ref': 'AD 290 - How will Staff Access be enforced for West 3', 'author': 'Former user (Deleted)', 'page_link': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590633118', 'page_ref': '590633118', 'tag': 'solution'} score: 0.4113536
{'ad_r

In [119]:
hits = client.search(
    collection_name="rnw_adr_collection",
    query_vector=embedding_model.encode("The current property claims process are fragmented and segmented with limited technology capability creating manual processes and limited data visibility . In turn this creates inefficiencies and customer dissatisfaction").tolist(),
    query_filter=models.Filter(**{
    "must": [
              {
                "key": "tag",
                "match": {
                  "any": [
                    "problem", 
                    "context"
                  ]
                }
              }
            ]        
    }),
    limit=15,
)
for hit in hits:
    print(hit.payload, "score:", hit.score)

In [167]:
hits = client.search(
    collection_name="rnw_adr_collection_3",
    query_vector=embedding_model.encode(" ").tolist(),
    query_filter=models.Filter(**{
    "must": [
              {
                "key": "tag",
                "match": {
                  "any": [
                    "problem", 
                    "context"
                  ]
                }
              }
            ]        
    }),
   limit=5,
)


{'ad_ref': 'AD 104 - Which system/s should map LOB codes, coverages etc. in the PC provided policy retrieve call consumed by claims systems?', 'author': 'Former user (Deleted)', 'full_adr_content': {'Analysis': ["{'column_headers': ['Criteria', 'Option 1PC does mapping', 'Option\\xa02BAPI does mapping', 'Option 3CC does mapping', 'Importance HIGH', 'Impact on upstream applications', 'Impact on downstream applications', 'Importance MEDIUM', 'Overall effort and reuse', 'Recommendation', ''], 'rows': []}", "{'column_headers': ['Criteria', 'Option 1PC does mapping', 'Option\\xa02BAPI does mapping', 'Option 3CC does mapping', 'Importance HIGH', 'Impact on upstream applications', 'Impact on downstream applications', 'Importance MEDIUM', 'Overall effort and reuse', 'Recommendation', ''], 'rows': []}", 'Option 1', 'PC does mapping', 'Option\xa02', 'BAPI does mapping', 'Option 3', 'CC does mapping', 'LOW', 'MEDIUM', 'HIGH', 'LOW', 'MEDIUM', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM', ''], 'Context': ['Th

In [174]:
for hit in hits:
    print( '=', hit.payload['subheading_content'], "=", "score:", hit.score)

=  = score: 0.99999994
=  = score: 0.99999994
=  = score: 0.99999994
=  = score: 0.99999994
=  = score: 0.99999994


In [68]:
# match it only with problem tags 

def top_n_hits(n, input_content_to_match, tags=[]):
    
    
    if tags:
        hits = client.search(
        collection_name="rnw_adr_collection_3",
        query_vector=embedding_model.encode(input_content_to_match).tolist(),
        query_filter=models.Filter(**{
                                    "must": [
                                              {
                                                "key": "tag",
                                                "match": {
                                                  "any": tags
                                                }
                                              }
                                            ]        
                                        }),
        limit=n,
        )
        return hits
    
    hits = client.search(
        collection_name="rnw_adr_collection_3",
        query_vector=embedding_model.encode(input_content_to_match).tolist(),
        limit = n
     )
    
    return hits



### Project Anchor

In [195]:
# testing sample Lean Canvases

project_anchor_problem_statement = "The current property claims process are fragmented and segmented with limited technology capability creating manual processes and limited data visibility . In turn this creates inefficiencies and customer dissatisfaction"


In [196]:
project_anchor_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =project_anchor_problem_statement , tags=["problem"])

In [197]:
for hit in project_anchor_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4586742
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4586742
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4502906
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4265918
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38196766


In [198]:
for hit in project_anchor_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Problem : Finance systems have evolved incrementally over time through acquisitions (CGU, WFI, etc.) and Organisational Changes (EO, Project Light). Consequently granular level of data needed to run the business in a way that delivers timely and accurate information is not captured from source systems. The Finance General Ledger is capturing and storing the information based on old structures, cost centres and legal entities requiring lots of transformation outside of SAP. Data related to insurance products are not stored in a consistent way. The data must be again transformed outside of SAP, using disconnected BI tools and source systems because reporting dimensions are restricted to Channel, Class and State. The Finance system has been heavily customized making it harder to maintain and limits the opportunities to leverage business partners to take on finance functions. Further, compliance to IFRS 17 will require granular level detail of policy and claims data that gets augmented wit

In [193]:
project_anchor_outcome = "Enable IAG to digitise its Property fulfilment process by implementing a Property Supply Chain management platform to seamlessly connect IAG customers and suppliers"

In [199]:
%time
project_anchor_outcome_hits_for_solution =  top_n_hits(n = 5, input_content_to_match =problem_statement_anchor , tags=["solution"])

CPU times: user 4 µs, sys: 3 µs, total: 7 µs
Wall time: 12.2 µs


In [200]:
for hit in project_anchor_outcome_hits_for_solution:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5503343
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.51444036
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49379465
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4626555
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4302451


In [206]:
for hit in project_anchor_outcome_hits_for_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Solution & key reason/s : Where required, claim summary history will be extracted from legacy claim system(s) during policy migration and stored in PolicyCenter. PolicyCenter will  supplement this with claim summary from ClaimCenter and any claims data declared as part of the quote for rating and underwriting. Pricing needs 3 years of claim history for rating. For brands/policies, where ClaimCenter is unable to provide this history, the data will be retrieved from legacy claim system(s) during policy migration and a static view will be stored in PolicyCenter. Additionally, PolicyCenter will retrieve claim summary information from ClaimCenter as & when required. The ClaimCenter 'getClaim' web service currently provides all the attributes required for pricing and underwriting for AU & NZ personal lines products. There may be a need to add additional attributes in the web service where there are specific requirements to transform data by applying business rules and these will be done in C

### Underwriting Restrictions - Sys Imp

In [79]:
underwriting_problem_st = '''Background
An underwriting restriction may be implemented in response to a major event or the risk of a major event occurring  such as;  Earthquake, Tsunami, Volcano, Wildfire, Ex-tropical cyclone.
An underwriting restriction is a restriction on the sale or increases in sum insured of risks within specific lines of business based on a geographical location. 
Problem
Due to the nature of events that trigger underwriting restrictions,  these need to be implemented as soon as possible after a decision has been made.  These events can happen outside of standard business hours.  
The IAG technical landscape is varied, complex and constantly changing and it is not clear how to get restrictions implemented or updated across brands, lines of business and geographical locations.   Anecdotally outside of  Personal lines State/Huon, State/AMI digital and AON CPF,  Portfolio have very little visibility on how to implement systems restrictions. 

 In the past getting system restrictions in place has been adhoc, using back channels rather than the Tech Front Door, with Portfolio team members reliant upon who they know rather than following  a prescribed Tech & Ops process. If the “go-to person” is not around then Portfolio may not know who to contact. '''

In [80]:
underwriting_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =underwriting_problem_st , tags=["problem"])

In [104]:
for hit in underwriting_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.48197982
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.45956102
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.43287438
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4211364
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41854545


In [81]:

for hit in underwriting_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Problem : AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? determined that the strategic solution for peril pricing and underwriting was to use the PRICE application. However this system is not ready for production yet. The peril underwriting can be delayed indefinitely, but it is necessary to have peril pricing from the first release. So what should the interim solution be for CGU Direct Home, Motor & Niche? 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523 
 AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche? 
 0.48197982 

Problem : When outbound correspondence for policy outbound is generated, a set of static documents is included.  This includes specified policy transaction inclusions such as PDS, SPDS, and KFS.  It may optionally include marketing inserts. The set of static documents can be derived from information passed into the document composition API (eg LoB, Brand, Tra

In [84]:
underwriting_problem_hits_problem

[ScoredPoint(id=30211537, version=440, score=0.48197982, payload={'ad_ref': 'AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche?', 'author': 'Former user (Deleted)', 'full_adr_content': {'Analysis': ["{'column_headers': ['Criteria', 'Option 1Source from PRICE', 'Option\\xa02Source from PDP', 'Option 3Source from FIT', 'Importance HIGH', 'Delivery Risk', 'Importance MEDIUM', 'Quality of Pricing', 'ReRecommendation', ''], 'rows': []}", "{'column_headers': ['Criteria', 'Option 1Source from PRICE', 'Option\\xa02Source from PDP', 'Option 3Source from FIT', 'Importance HIGH', 'Delivery Risk', 'Importance MEDIUM', 'Quality of Pricing', 'ReRecommendation', ''], 'rows': []}", 'Option 1', 'Source from PRICE', 'Option\xa02', 'Source from PDP', 'Option 3', 'Source from FIT', 'HIGH', "We flipped from PRICE because it's alleged to not be ready. Sourcing static data from PRICE requires additional work, and is riskier and less functionally rich t

In [88]:
underwriting_problem_hits_problem[0].id

30211537

In [90]:
underwriting_problem_hits_problem[0].version

440

In [91]:
underwriting_problem_hits_problem[0].score

0.48197982

In [97]:
print(underwriting_problem_hits_problem[0].vector)
print(underwriting_problem_hits_problem[0].shard_key)

None
None


In [92]:
underwriting_problem_hits_problem[0].payload

{'ad_ref': 'AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche?',
 'author': 'Former user (Deleted)',
 'full_adr_content': {'Analysis': ["{'column_headers': ['Criteria', 'Option 1Source from PRICE', 'Option\\xa02Source from PDP', 'Option 3Source from FIT', 'Importance HIGH', 'Delivery Risk', 'Importance MEDIUM', 'Quality of Pricing', 'ReRecommendation', ''], 'rows': []}",
   "{'column_headers': ['Criteria', 'Option 1Source from PRICE', 'Option\\xa02Source from PDP', 'Option 3Source from FIT', 'Importance HIGH', 'Delivery Risk', 'Importance MEDIUM', 'Quality of Pricing', 'ReRecommendation', ''], 'rows': []}",
   'Option 1',
   'Source from PRICE',
   'Option\xa02',
   'Source from PDP',
   'Option 3',
   'Source from FIT',
   'HIGH',
   "We flipped from PRICE because it's alleged to not be ready. Sourcing static data from PRICE requires additional work, and is riskier and less functionally rich than the original strategic solution.

In [98]:
underwriting_problem_hits_problem[0].payload["subheading_content"]

'Problem : AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? determined that the strategic solution for peril pricing and underwriting was to use the PRICE application. However this system is not ready for production yet. The peril underwriting can be delayed indefinitely, but it is necessary to have peril pricing from the first release. So what should the interim solution be for CGU Direct Home, Motor & Niche?'

In [99]:
underwriting_problem_hits_problem[0].payload["tag"]

'problem'

In [100]:
underwriting_problem_hits_problem[0].payload["ad_ref"]

'AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche?'

In [101]:
underwriting_problem_hits_problem[0].payload["page_ref"]

'590260523'

In [102]:
underwriting_problem_hits_problem[0].payload["page_link"]

'https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523'

In [106]:
results = {}
for hit in underwriting_problem_hits_problem: 
    value = hit.payload["page_link"] + '\n' + hit.payload["subheading_content"]
    results.update({ hit.payload["ad_ref"] : value })

print(results)

{'AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche?': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523\nProblem : AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? determined that the strategic solution for peril pricing and underwriting was to use the PRICE application. However this system is not ready for production yet. The peril underwriting can be delayed indefinitely, but it is necessary to have peril pricing from the first release. So what should the interim solution be for CGU Direct Home, Motor & Niche?', 'AD 108 - Where should correspondence insert/onsert management be done?': 'https://iag.atlassian.net/wiki/spaces/SEP/pages/590179605\nProblem : When outbound correspondence for policy outbound is generated, a set of static documents is included.\xa0 This includes specified policy transaction inclusions such as PDS, SPDS, and KFS.\xa0 It may optionally include\xa0marketi

In [107]:
import json
with open ("vdb3.json", "w") as jsonfile:
    json.dump(results, jsonfile)

In [113]:
demo_problem = """
Problem
Due to the nature of events that trigger underwriting restrictions,  these need to be implemented as soon as possible after a decision has been made.  These events can happen outside of standard business hours.  
The IAG technical landscape is varied, complex and constantly changing and it is not clear how to get restrictions implemented or updated across brands, lines of business and geographical locations.   Anecdotally outside of  Personal lines State/Huon, State/AMI digital and AON CPF,  Portfolio have very little visibility on how to implement systems restrictions. 

 In the past getting system restrictions in place has been adhoc, using back channels rather than the Tech Front Door, with Portfolio team members reliant upon who they know rather than following  a prescribed Tech & Ops process. If the “go-to person” is not around then Portfolio may not know who to contact.
"""

In [114]:
demo_context = """
Background
An underwriting restriction may be implemented in response to a major event or the risk of a major event occurring  such as;  Earthquake, Tsunami, Volcano, Wildfire, Ex-tropical cyclone.
An underwriting restriction is a restriction on the sale or increases in sum insured of risks within specific lines of business based on a geographical location. 
"""

In [116]:
demo_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =demo_problem , tags=["problem"])
for hit in demo_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Problem : When outbound correspondence for policy outbound is generated, a set of static documents is included.  This includes specified policy transaction inclusions such as PDS, SPDS, and KFS.  It may optionally include marketing inserts. The set of static documents can be derived from information passed into the document composition API (eg LoB, Brand, Transaction type).  The set of inclusions will periodically change via a business process, for example when a new PDS is issued, or a new marketing campaign occurs Currently within the HUON eco-system this is managed within custom mainframe code. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590179605 
 AD 108 - Where should correspondence insert/onsert management be done? 
 0.45993492 

Problem : The systems required for policy sales and service across IAG are being simplified into one ecosystem as part of the Serenity Policy program. To provide the required business capability, an existing IAG system will be reused or new techno

In [117]:
demo_context_hits_problem =  top_n_hits(n = 5, input_content_to_match =demo_context , tags=["context"])
for hit in demo_context_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Context : Refer AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523 
 AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche? 
 0.5387314 

Context : Embargoes applications are used to create an embargo in regions (in AU and NZ) for specific products so that calling apps can check if a policy creation needs to be refused/endorsed due to a severe event in that area (Storm, Flood, Fire etc.). While the existing CL Embargoes service provides the ability to underwrite at postcode, suburb and town level, there are more granular requirements from NZ team. The solution is expected to support underwriting based on coordinates(x,y) and also provide the ability to Create and apply Restriction zone maps for geographical areas subject to tightened underwriting controls such as, referral requirements for homes identified in coastal erosion locations. 

In [111]:
1

1

In [ ]:
k = {"d":"dfd", "f":"dfdfg"}
print(k.keys())

In [ ]:
print

In [209]:
underwriting_problem_hits_context =  top_n_hits(n = 5, input_content_to_match =underwriting_problem_st , tags=["context"]) # since the problem st includes a background as well

In [210]:
for hit in underwriting_problem_hits_context:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5200924
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5120088
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.51148623
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49802446
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49030608


In [211]:
for hit in underwriting_problem_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Context : Embargoes applications are used to create an embargo in regions (in AU and NZ) for specific products so that calling apps can check if a policy creation needs to be refused/endorsed due to a severe event in that area (Storm, Flood, Fire etc.). While the existing CL Embargoes service provides the ability to underwrite at postcode, suburb and town level, there are more granular requirements from NZ team. The solution is expected to support underwriting based on coordinates(x,y) and also provide the ability to Create and apply Restriction zone maps for geographical areas subject to tightened underwriting controls such as, referral requirements for homes identified in coastal erosion locations. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590305735 
 AD 154 - Which system/s will perform embargoes? 
 0.5200924 

Context : Refer AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523 

In [212]:
underwriting_solution = '''We need a baseline process immediately which means an  understanding of our current state,  what is possible and what is not possible in our systems.  This will enable Portfolio to put some manual processes in place where necessary, to prevent the sale of risks.
 From there we need to understand the opportunities for improvement, including the strategic solution (EPD = Embargo Tool) what the gaps are and how we can resolve them.

For example, some systems have automatic referral triggers and some don’t, but there isn’t clear visibility on this or how long it would take to get the automatic referral triggers updated.  

There is an ongoing requirement to have a Tech and Ops owner for this process so that Portfolio can continue to liase with them to review and keep the process up-to-date. '''

In [213]:
underwriting_solution_hits_solution =  top_n_hits(n = 5, input_content_to_match =underwriting_solution , tags=["solution"]) # since the problem st includes a background as well

In [214]:
for hit in underwriting_solution_hits_solution:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46292618
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3938014
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3932264
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3691886
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36720714


In [215]:
for hit in underwriting_solution_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Solution & key reason/s : Use existing Customer Labs Embargo service with uplift as required. Key reasons include: 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590089023 
 AD 022 - Which system/s will perform embargoes for CGU Direct Home, Motor & Niche? 
 0.46292618 

Solution & key reason/s : VRD - the incumbent application currently supporting HUON, PMS, and ClaimCenter. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590107774 
 AD 056 - What system/s will store inbound and outbound documents for CGU Direct Home and Motor? 
 0.3938014 

Solution & key reason/s : Incumbent application supporting HUON / CRODS tracking to be retained / enhanced DMS Outbound Value Chain: 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590122837 
 AD 011 - What system/s will provide correspondence tracking? 
 0.3932264 

Solution & key reason/s : PolicyCenter and BillingCenter will enable SAP ECC6 to generate general ledger posting records including calculation of earned premium. The solution d

### Quote consent uplift 

In [216]:
quote_problem_st = ''' 
'New and existing customers who obtain a quote with NRMA are asked whether they consent to be followed up about their quote. This consent enables Marketing to send emails to the customer offering help with their quote and prompting them to purchase.

At present, this consent is limited to follow ups about that specific quote, and the consent expires once the quote validity period ends.

This proposal seeks to expand the scope of quote consent to enable two specific scenarios:1) Ability to send a pre-populated quote to the customer one year after they obtained their quote. This would allow customers to easily compare NRMA with their chosen insurer.
2) Ability to send targeted relevant offers to a customer for a period of up to two years. For example, if the customer obtains a quote for motor insurance, and then later NRMA has a ‘$100 off’ motor offer.

We currently operate at a competitive disadvantage to our competitors who already do this and are eroding our market share
'''

In [218]:
quote_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =quote_problem_st , tags=["problem"]) # since the problem st includes a background as well

In [219]:
for hit in quote_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38785994
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38056582
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.37043795
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36423054
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36182532


In [220]:
for hit in quote_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Problem : How will we allow customers and colleagues to access, update and sync. party and policy details while they are customer/s to both legacy and purple (Serenity) business. Example use cases: Use Case 1: SGI customer located in WA/SA: Use Case 2: NRMA customer located in NSW : A decision is required around the customer and colleagues experience where both new and Legacy business are applicable, ensuring we invest proportional to the business value. Key thing to note here is that transition state when customers can potentially have both new and legacy business can last for a long time, as commercial business and CTP are not slated 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119195 
 AD 063 - How will policy servicing through CEP support co-existence of PolicyCenter and legacy policy systems? 
 0.38785994 

Problem : How will we allow customers and colleagues to access, update and sync. party and policy details while they are customer/s to both legacy and purple (Serenity)

In [217]:
quote_outcome = '''
"Uplift quote UI in Q&B and PC
Update to a major version of consent to include quote follow-up, pre-populated quotes and storage of prospect customers for 2 yrs
The ability for authenticated and prospect customers to unsubscribe from quote consent and special offers (Marketing)
Omnichannel solution for QRC"
'''

In [221]:
quote_outcome_hits_solution =  top_n_hits(n = 5, input_content_to_match =quote_outcome , tags=["solution"]) # since the problem st includes a background as well

In [222]:
for hit in quote_outcome_hits_solution:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5591235
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5199705
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.44841117
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41993284
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41796753


In [223]:
for hit in quote_outcome_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Solution & key reason/s : Below is the key diagram from Customer Tech wiki  2.2.3.15 Identified Quote Flow 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590561163 
 AD 260 - How are Identified Customers redirected post login 
 0.5591235 

Solution & key reason/s : When Customers save Quotes through Self Service or Assisted channels they will be sent an email with a URL link that retrieves their Quote. The Customer provides the email address as time of saving the Quote. PC will send the Quote information to the Document BAPI that will fetch a URL link, from a Link Generator service that Digital will build, before sending the link, email and Quote information to SmartComms for delivery. Methods for retrieving a Quote without an email link are out of scope for release 1 (

    

            




    
 SNP-3333
                    -
            Getting issue details...
STATUS

). The solution will be covered in AD 143 - How will Customers save and retrieve Quotes in RX? 
 https://iag.a

In [224]:
quote_outcome_hits_context =  top_n_hits(n = 5, input_content_to_match =quote_outcome , tags=["context"]) # since the problem st includes a background as well

In [225]:
for hit in quote_outcome_hits_context:
    print(hit.payload.keys(), hit.score)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.466308
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39984286
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3972297
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39354405
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38396728


In [261]:
for hit in quote_outcome_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

Context : The Candidate Scope Document has the following requirements: However, the ability to retrieve quotes requires, as a higher priority from a customer experience standpoint, a URL link to the Quote to be emailed to the Customer as opposed to the Customer finding the page on the internet themselves and entering their Quote number and passing a OTP or PII data entry challenge. Refer AD 138 - How will Customers save and retrieve Quotes for CGU Direct Home, Motor & Niche? for more details. When it comes to saved anonymous quotes, which are important for conversion rates especially through  the digital channel (can this be verified?), the candidate scope document is saying that emails cannot be sent and the alternative OTP or PII data entry method must be used in order to satisfy the other requirements that quotes must be retrievable. This work, for OTP or PII data entry, is expected to require more time and effort than just extending the identified ("non-anonymous") email solution f

In [268]:
act_ctp_problem = """ 
In October 2018 it is expected that the ACT CTP scheme reform will be published. Included in this reform will be timeframes in which Insurers need to implement the reform changes. It is likely that reform requirements will need changes in the Claims Platform, Digital Platform and Data and reporting. The opportunity cost of not meeting the reform requirements may prevent IAG from operating in the ACT scheme.
"""

act_ctp_outcome = """
"WE BELIEVE THAT
Consolidation of all CTP Claims (NSW, ACT and QLD) onto a single Claims platform and automation of outbound correspondence
FOR
CTP Claims
WILL ENABLE
Improved efficiencies and tighter control of processing and decision making
WE WILL KNOW THIS TO BE TRUE WHEN WE SEE
More timely assistance to injured people's health recovery and return them, as much as possible,
to their previous levels of participation.
Increased capacity for Claims consultants (more files managed per person)
Reduced claims leakage

Effort (Man Days): XX days
Total Cost Estimate of Initiative (including all infrastructure etc): $XX"
"""

In [264]:
act_ctp_outcome.strip() 

'"WE BELIEVE THAT\nConsolidation of all CTP Claims (NSW, ACT and QLD) onto a single Claims platform and automation of outbound correspondence\nFOR\nCTP Claims\nWILL ENABLE\nImproved efficiencies and tighter control of processing and decision making\nWE WILL KNOW THIS TO BE TRUE WHEN WE SEE\nMore timely assistance to injured people\'s health recovery and return them, as much as possible,\nto their previous levels of participation.\nIncreased capacity for Claims consultants (more files managed per person)\nReduced claims leakage\n\nEffort (Man Days): XX days\nTotal Cost Estimate of Initiative (including all infrastructure etc): $XX"'

In [265]:
1

1

# 

In [266]:
act_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =act_ctp_problem.strip() , tags=["problem"]) # since the problem st includes a background as well

for hit in act_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

for hit in act_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47661442
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4185539
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41360688
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.407043
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.40460297
Problem : IAG is moving its legacy policy systems to new source system i.e. Guidewire policy centre (PC). During this transition and up to the point in time where all policy holders have been moved to the PC, reporting needs will need to met by brining data from different systems together (co-existence) (Transactions/Historic). 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119177 
 AD 061 - How will reportin

In [273]:
1

1

# act_problem_hits_context =  top_n_hits(n = 5, input_content_to_match =act_ctp_problem.strip() , tags=["context"]) # since the problem st includes a background as well

for hit in act_problem_hits_context:
    print(hit.payload.keys(), hit.score)

for hit in act_problem_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

In [ ]:
1

In [275]:
1

1

In [276]:
1

1

In [272]:
act_outcome_hits_solution =  top_n_hits(n = 5, input_content_to_match =act_ctp_outcome.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in act_outcome_hits_solution:
    print(hit.payload.keys(), hit.score)

for hit in act_outcome_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5126008
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46334475
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4552157
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.44478792
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41815546
Solution & key reason/s : PC will expose a policy retrieve API that matches the current implementation across existing policy systems in AU & NZ. This is for the same reasons as in AD 031 for the "policy retrieve" function where transformation is in the provider and not repeated in multiple consumers. In NZ currently, the API is used for digital claims however the native interface where CC does the mapping is util

In [295]:
consumer_pro_problem = """
"The final report of the Financial Service Royal Commission has resulted in a series of legislation changes relevant to IAG. This proposal deals with:
Anti Hawking: The intent is to stop pressure selling, particularly driven by sales tactics in life insurance.  Existing Anti Hawking Provisions were found to not adequately protect consumers. New Bill offers more extensive protection, prohibiting offers to sell or issue financial products in the course of, or because of, unsolicited contact. The hawking prohibitions aim to prevent pressure selling and marketing techniques that may detract from clients’ decision making
Deferred Sales Model (DSM): The intent is to allow customers to consider before buying insurance products that might represent poor value for them.Recommendation for the implementation of industry wide deferred sales model for add on insurance. New Bill contains a four day deferred sales period

Final legislation has not been released yet (as per 12/10/20) – a number of industry submissions were made, but outcome uncertain"
"""

In [296]:
cpro_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =act_ctp_outcome.strip() , tags=["problem"]) # since the problem st includes a background as well

for hit in cpro_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

for hit in cpro_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39660856
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39219457
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38976735
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.34975412
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3457815
Problem : IAG is moving its legacy policy systems to new source system i.e. Guidewire policy centre (PC). During this transition and up to the point in time where all policy holders have been moved to the PC, reporting needs will need to met by brining data from different systems together (co-existence) (Transactions/Historic). 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119177 
 AD 061 - How will report

In [280]:
1

1

In [297]:
cpro_sol_concept = """
"Risk based ‘minimum solution’ for 1/01/21 based on spirit of legislation
Stop practices that have a high likelihood to be unacceptable going forward (following a rating process)
Refine solution for Anti Hawking further once final legislation known and roll out in further release(s)
Develop DSM solution based on final legislation
Risk of non-compliance for Anti Hawking
Avoiding re-work
Minimised disruption for business and business partners
Opportunity to take into account the right strategic considerations "
"""

In [298]:
cpro_sol_hits_solution =  top_n_hits(n = 5, input_content_to_match =cpro_sol_concept.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in cpro_sol_hits_solution:
    print(hit.payload.keys(), hit.score)

for hit in cpro_sol_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.44613102
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4098643
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.40404016
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39760262
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39706033
Solution & key reason/s : The strategic preference is obviously to pursue small scale MFE deployment in CEP as early as possible to reduce the burden and risk in the following NRMA East release where MFE's will be mandatory to deliver against the scaled requirements of a high volume contact centre. This option however appears to be infeasible from a delivery perspective as it would require build of 2 MFEs plus th

In [283]:
1

1

In [299]:
cpro_sol_hits_context =  top_n_hits(n = 5, input_content_to_match =cpro_sol_concept.strip() , tags=["context"]) # since the problem st includes a background as well

for hit in cpro_sol_hits_context:
    print(hit.payload.keys(), hit.score)

for hit in cpro_sol_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46216556
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.43696302
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.37267834
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36826116
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36410072
Context : Refer AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590260523 
 AD 136 - Which system/s will provide operational peril risk for pricing for CGU Direct Home, Motor & Niche? 
 0.46216556 

Context : There is a large amount of technical change risk occurring in the policy sales and service ecosystem as part of Se

In [290]:
1

1

In [300]:
anon_problem = """
" There is a business opportunity to drive growth using targeted marking opportunities designed to appeal to the subset of customers who 
are common to the NRMA motoring association. IAG and the NRMA have a mechanism for matching customer data however the current method is 
error prone and is not completely accurate. This represents business and reputational risk as customers may miss out on marketing offers they are 
entitled to or may be offered offers they are not entitled to. "
"""

In [306]:
anon_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =anon_problem.strip() , tags=["problem"]) # since the problem st includes a background as well

for hit in anon_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

for hit in anon_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.44660783
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4402598
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4290794
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4242695
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.42014033
Problem : In order to provide an enhanced customer experience the IVR and Telephony systems within IAG need to be able to interact with the claims and policy systems in order to provide customer ID&V services and appropriately route calls to the ideal consultants whilst providing a tailored customer experience. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119185 
 AD 073 - Which system/s will provide Teleph

In [301]:
1

1

In [307]:
anon_problem_hits_context =  top_n_hits(n = 5, input_content_to_match =anon_problem.strip() , tags=["context"]) # since the problem st includes a background as well

for hit in anon_problem_hits_context:
    print(hit.payload.keys(), hit.score)

for hit in anon_problem_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5327083
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49448836
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49448836
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49046013
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46947408
Context : The outcome of to AD 164 - How many mobile apps will be built for all brands? is that the existing NRMA app will be used for the IAG "Purple" app for all brands. The "Purple" Native app currently supports NRMA Blue, which for Serenity R3, will need to co-exist with NRMA Silver QLD (home, motor, niche), until all of NRMA blue is migrated from Huon to PolicyCenter - see AD 208 - How will all the Customer 

In [303]:
1

1

In [304]:
1

1

In [309]:
anon_solution_concept = """
" Anonflow is a privacy preserving identity solution designed to allow organisations to share features of common customers without directly 
sharing personally identifiable information where appropriate consent has been provided. 
The solution allows the matching of identities between IAG and its partners, in this case NRMA, in a way that is accurate but does not require the 
direct sharing of PII information. Customer ""feature data"" can then be shared based on an accurate view of common customers"
"""

In [310]:
anon_sol_hits_sol =  top_n_hits(n = 5, input_content_to_match =anon_solution_concept.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in anon_sol_hits_sol:
    print(hit.payload.keys(), hit.score)

for hit in anon_sol_hits_sol:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.51744133
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.50634944
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47734132
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47609237
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46912217
Solution & key reason/s : On most entry points the Customer selects their State and sometimes the Product so they can be routed appropriately or given a message. For a detailed view of all the self-service entry points, refer to E2E - Customer self-service entry points For a detailed list of the solution for each entry point, refer to the "Options, considerations & solutions" section below. Options, consideratio

In [ ]:
anon

In [311]:
1

1

In [324]:
asb_problem = """
"Sales volumes have dropped from bank partners as staff are
un-incentivised and less motivated. Online bank leads are best converted
by IAG call centre personnel. Cancellations have increased and 
front line staff are not best equipped 
to handle them"
"""

In [325]:
asb_probleml_hits_problem =  top_n_hits(n = 5, input_content_to_match =asb_problem.strip() , tags=["problem"]) # since the problem st includes a background as well

for hit in asb_probleml_hits_problem:
    print(hit.payload.keys(), hit.score)

for hit in asb_probleml_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41767615
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3694591
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.35530046
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.3550286
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.32511282
Problem : In order to provide an enhanced customer experience the IVR and Telephony systems within IAG need to be able to interact with the claims and policy systems in order to provide customer ID&V services and appropriately route calls to the ideal consultants whilst providing a tailored customer experience. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119185 
 AD 073 - Which system/s will provide Telep

In [314]:
1

1

In [315]:
1

1

In [326]:
asb_sol_con = """
"Chemistry has sourced a tried and tested ‘plug-in’ solution. It connects bank quote enquiries to IAG contact centre personnel.
It connects policy cancellation requests to the IAG specialised Save Team. "
"""

In [327]:
asb_sol_con_hits_context =  top_n_hits(n = 5, input_content_to_match =asb_sol_con.strip() , tags=["context"]) # since the problem st includes a background as well

for hit in asb_sol_con_hits_context:
    print(hit.payload.keys(), hit.score)

for hit in asb_sol_con_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.57023084
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49254507
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.48053664
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47196895
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47046053
Context : IAG needs to have a solution that will satisfy the following requirement: 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590119187 
 AD 074 - Which system/s will provide complaints management capability for R1? 
 0.57023084 

Context : The policy transaction presents the customer with an option to specify whether the Asset in question in financed or not. If the asset is financed, the customer is req

In [318]:
¡

SyntaxError: invalid character '¡' (U+00A1) (4217278722.py, line 1)

In [319]:
¡

SyntaxError: invalid character '¡' (U+00A1) (4217278722.py, line 1)

1

In [328]:
asb_outcome = """
"Removing friction in the enquiry process can increase lead generation by 47%
Brand consideration due to speed of response
More speed in the sales funnel can increase conversion by as much as 72%"
"""

In [329]:
asb_outcome_hits_solution =  top_n_hits(n = 5, input_content_to_match =asb_outcome.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in asb_outcome_hits_solution:
    print(hit.payload.keys(), hit.score)

for hit in asb_outcome_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.35843068
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.32608727
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.31441867
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.31441867
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.31441867
Solution & key reason/s : Option 1 - Source peril pricing and factors directly from the PRICE application. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590233543 
 AD 125 - Which system/s will provide operational peril risk for pricing and underwriting? 
 0.35843068 

Solution & key reason/s : For a more consumable view of the solution, see AD136 - Peril Pricing strategic solution overview 
 https://iag.atl

In [323]:
1

1

In [341]:
com_cc_problem = """
"Custom Complaints screens have been built into to ClaimCenter AU (CC8), these screens store data in the CC8 database, and a UI-based robot takes this data and automates Orbit Complaints Management to create complaints.

The robot routinely breaks is very costly to update ($200k+) and generates many exceptions.  After further investigation, the robot is not completely at fault, there is often erroneous or missing data which causes failures.  Enhancements have been made to CC8 to improve data quality, but exceptions are still occurring.

Due to the high number of exceptions, the robot has been turned off, Claims consultants are now using Orbit directly to lodge complaints.  The consultants have expressed a strong desire not to use Orbit; they are opposed to adopting another system and prefer to remain within CC8.  Additionally, using Orbit for complaints incurs significant time costs due to the need to re-enter information already present in CC8."
"""

In [342]:
com_cc_problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =com_cc_problem.strip() , tags=["problem"]) # since the problem st includes a background as well

for hit in com_cc_problem_hits_problem:
    print(hit.payload.keys(), hit.score)

for hit in com_cc_problem_hits_problem:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.488239
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4707734
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.43395942
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4094862
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.40458688
Problem : The Complaints Management was not in scope for the serenity program as the expectation was that it will be handled by Complaints Program. As the CCM will not be ready for the R1 go live, the tactical solution will be required to enable the capability for NRMA Silver consultants.  Solution & key reason/s Orbit Customer Complaints Management will be enabled for the NRMA Silver consultants 
 https://iag.atlas

In [347]:
1

1

In [351]:
com_cc_problem_hits_context =  top_n_hits(n = 5, input_content_to_match =com_cc_problem.strip() , tags=["context"]) # since the problem st includes a background as well

for hit in com_cc_problem_hits_context:
    print(hit.payload.keys(), hit.score)

for hit in com_cc_problem_hits_context:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5231303
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4892717
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.45023447
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.41963643
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4043826
Context : The initial solution assumed that all customer maintenance, for both CGU SME and CGU Personal line customers, will be carried out via Orbit. With the recent decision to exclude Orbit from CGU Release1, it became important to ensure that Orbit still continues to service CGU SME customers without being affected by the introduction of CGU Personal line customers. Likewise, it was important to ensure that Pol

In [349]:
1

1

In [350]:
1

1

In [333]:
1

1

In [343]:
com_cc_outcome  = """
"Removes complaints management process from CC8 which helps with our move to GWCP.
Regulatory changes to the complaints process occur reasonably frequently, the Complaints MFE team is close to the complaints domain and will make the necessary changes (in one place to support multiple consumers).
Complaints MFE uplift, to support claims related complaints and party management, has the potential to be beneficial to other users in the future (e.g. Partners)."
"""


In [344]:
com_cc_outcome_hits_solution =  top_n_hits(n = 5, input_content_to_match =com_cc_outcome.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in com_cc_outcome_hits_solution:
    print(hit.payload.keys(), hit.score)

for hit in com_cc_outcome_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.40032095
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.37116227
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36740196
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.36494416
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.35897887
Solution & key reason/s : The strategic preference is obviously to pursue small scale MFE deployment in CEP as early as possible to reduce the burden and risk in the following NRMA East release where MFE's will be mandatory to deliver against the scaled requirements of a high volume contact centre. This option however appears to be infeasible from a delivery perspective as it would require build of 2 MFEs plus t

In [336]:
1

1

In [345]:
com_cc_sol_con = """
"With the desire to stay in CC8, and create a more integrated (real-time) solution, two options have been explored:
Option 1: Replace the custom solution with the Complaints MFE - Recommended
Option 2: Update the custom screens to use Complaints APIs"
"""

In [346]:
com_cc_sol_con_hits_solution =  top_n_hits(n = 5, input_content_to_match =com_cc_sol_con.strip() , tags=["solution"]) # since the problem st includes a background as well

for hit in com_cc_sol_con_hits_solution:
    print(hit.payload.keys(), hit.score)

for hit in com_cc_sol_con_hits_solution:
    print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39933097
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.39419124
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.37508819
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.35593525
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.34796685
Solution & key reason/s : Option 1, the centralised notification pattern, has been selected as it delivers a consistent, compliant solution and does not require multiple front ends to all achieve compliant solutions separately. 
 https://iag.atlassian.net/wiki/spaces/SEP/pages/590699604 
 AD 299 - Customer Notifications For Contact Details Updates 
 0.39933097 

Solution & key reason/s : Due to delivery timeline

In [ ]:
### Automate hits , results for a list of input<>tag pairs 



In [361]:
problems = [
"""
"A key component of enabling an API-led ecosystem is the availability of an API Developer Portal, where API consumers can view an API catalogue, view API documentation, register to use and generally understand how the API offerings can be consumed.  Currently, IAG Australia and New Zealand use different solutions, none of which truly meet the emerging needs for both a customer-facing Developer Portal, nor the needs of our internal API consumers.
Architecturally, the API Developer Portal should act as a central marshalling point for API consumers, who after registration can then be directed to the appropriate endpoint for API consumption.  Because of this, an ideal solution would be to define a singular API Developer Portal solution (usable for both AU and NZ use cases) which can interact with a range of API Gateway capabilities, providing consumers with a singular user experience and API delivery teams with options in terms of API exposure."
"""
]

In [362]:
sol_cons = [
"""
"Proposed solution is to use Open source AWS Developer portal as a stop gap until the Target state of AWS Api Management is available in the market.

This option provides an architecture which is considered best practice and allows the option where the required extensions IAG has to make to the dev portal to bring it up to the target state and that could be contributed back to the open source project to become part of the product."
""", 
    """
    "Proposed solution is to use Open source AWS Developer portal as a stop gap until the Target state of AWS Api Management is available in the market.

This option provides an architecture which is considered best practice and allows the option where the required extensions IAG has to make to the dev portal to bring it up to the target state and that could be contributed back to the open source project to become part of the product."
    """
]

In [363]:
outcomes = [
"""
"It enables developers  and  architects know what APIs are available within a domain so they do not replicate the work. 
It enables accountability for the APIs. 
It would provide a single platform for developers to share their domain APIs and allow the business to view their capability."
""",
"""
"Technology advancements to improve delivery of the Distribution CVP within the retail network  and customer journey.

Improve the customer journey experience delivering a digital first focus.
Establish new cutomer experience and digital leadership.
Enhances customer experience  and uplifts digital capability of customer and our people."
"""

]

In [368]:


for problem in problems : 
    problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =problem.strip() , tags=["problem"]) # since the problem st includes a background as well

    for hit in problem_hits_problem:
        print(hit.payload.keys(), hit.score)
    
    for hit in problem_hits_problem:
        print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')



dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.68173456
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49629495
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.47557086
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46150455
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.38936007
Problem : Digital Systems of Engagement developed by Digital for Serenity Policy will be developed using the Single-Page Application (SPA) architecture, which manifests in client-side initiated calls being the predominant use of business APIs within this solution.  From a routing, access and security perspective, these Digital systems are "outside the network", requiring any APIs consumed by these systems to be 

In [357]:
¡

SyntaxError: invalid character '¡' (U+00A1) (4217278722.py, line 1)

In [372]:

for problem in problems : 
    problem_hits_context =  top_n_hits(n = 5, input_content_to_match =problem.strip() , tags=["context"]) # since the problem st includes a background as well

    for hit in problem_hits_context:
        print(hit.payload.keys(), hit.score)
    
    for hit in problem_hits_context:
        print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.61475646
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5079618
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5045784
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.502586
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5024046
Context : To support IAG's strategy of becoming digital-first and more API driven, we will be exposing APIs to be consumed by external parties, including: Ideally, when one of our external-facing APIs is invoked, we will know: These tokens are used to: Within this context, the AD aims to present solution options that address on what platform will issue M2M System Tokens and User Tokens to external API B2B application

In [369]:
for sol in sol_cons : 
    problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =sol.strip() , tags=["solution"]) # since the problem st includes a background as well

    for hit in problem_hits_problem:
        print(hit.payload.keys(), hit.score)
    
    for hit in problem_hits_problem:
        print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

    print("="*80)



dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.62814367
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4942394
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.49221808
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4574891
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.4314949
Solution & key reason/s : Use a Cloud Native API Gateway (Amazon API Gateway) for external Systems of Engagement. Due to the AWS-centric design of the Serenity Digital platform, the extremely low costs of the Amazon API Gateway and the simplicity of the use case, Amazon API Gateway is the most suitable technology for Serenity Policy systems of engagement, specifically the Single Page Applications and mobile apps in

In [364]:
1

1

In [365]:
1

1

In [370]:
for outcome in outcomes : 
    problem_hits_problem =  top_n_hits(n = 5, input_content_to_match =outcome.strip() , tags=["solution"]) # since the problem st includes a background as well

    for hit in problem_hits_problem:
        print(hit.payload.keys(), hit.score)
    
    for hit in problem_hits_problem:
        print(hit.payload["subheading_content"],'\n', hit.payload["page_link"],'\n',hit.payload["ad_ref"],'\n', hit.score, '\n')

    print("="*80)

dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.53357387
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.5138458
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.48474288
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.46412864
dict_keys(['ad_ref', 'author', 'full_adr_content', 'page_link', 'page_ref', 'subheading_content', 'tag']) 0.43868968
Solution & key reason/s : The Edge API Framework will be used to build Exposure APIs for business logic from PolicyCenter and BillingCenter as there is greater coverage of business capability that can be leveraged. It will also allow easier adoption of GW Sales & Service Portals should that be a future need.  Selection of SOAP can be made by the appropriate architect within the specific context of each integratio

In [11]:
1

1

### Understanding structure of vdb

In [13]:
# Function to check the number of documents for a specific metadata filter
def count_documents_with_metadata_filter(collection_name, filter_key, filter_value):
    # Create a filter for the specific metadata key and value
    query_filter = Filter(
        must=[
            Match(key=filter_key, value=filter_value)
        ]
    )
    # Count the number of documents matching the filter
    count = client.count(collection_name, query_filter)
    return count

In [17]:
def count_documents_with_metadata_filter2(collection_name, filter_key, filter_value):
    # Create a filter for the specific metadata key and value
    query_filter=models.Filter(**{
                                    "must": [
                                              {
                                                "key": filter_key,
                                                "match": {
                                                  "any": [filter_value]
                                                }
                                              }
                                            ]        
                                        })
    # Count the number of documents matching the filter
    count = client.count(collection_name, query_filter)
    return count



In [18]:
# Example usage:
collection_name = "rnw_adr_collection_3"

# 1. Count documents with specific metadata filter
# specific_metadata_count = count_documents_with_metadata_filter(collection_name, "tag", "problem")

specific_metadata_count = count_documents_with_metadata_filter2(collection_name, "tag", "problem")


print(f"Count of documents with specific metadata filter: {specific_metadata_count}")


Count of documents with specific metadata filter: count=285


In [19]:
specific_metadata_count

CountResult(count=285)

In [60]:
type(specific_metadata_count)

qdrant_client.http.models.models.CountResult

In [62]:
specific_metadata_count = count_documents_with_metadata_filter2(collection_name, "tag", "context")

specific_metadata_count

CountResult(count=285)

In [36]:
# Function to check the number of documents for multiple metadata filters
def count_documents_with_multiple_metadata_filters(collection_name, filters):
    # Create a list of filters for each key-value pair
    filter_conditions = [models.Match(key=k, value=v) for k, v in filters.items()]
    query_filter = models.Filter(must=filter_conditions)
    # Count the number of documents matching the filters
    count = client.count(collection_name, query_filter)
    return count


In [44]:
# Function to check the number of documents for multiple metadata filters
def count_documents_with_multiple_metadata_filters2(collection_name, filters):
    # Create a list of filters for each key-value pair
    filters_con = [{ "key": key, "match": { "any": [value] }} for key, value in filters.items()]
    query_filter=models.Filter(**{
                                    "must": filters_con    
                                        })
     # Count the number of documents matching the filters
    count = client.count(collection_name, query_filter)
    return count



In [46]:
# 2. Count documents with multiple metadata filters
multiple_metadata_filters = {"tag": "problem",  "author": "Former user (Deleted)"}
multiple_metadata_count = count_documents_with_multiple_metadata_filters2(collection_name, multiple_metadata_filters)
print(f"Count of documents with multiple metadata filters: {multiple_metadata_count}")

Count of documents with multiple metadata filters: count=178


In [42]:
# filters = [{ "key": key, "match": { "any": [value] }} for key, value in filters.items()]


query_filter=models.Filter(**{
                                    "must": [
                                        { "key": "tag", "match": { "any": ["problem"] } },
                                        { "key": "author", "match": { "any": ["Former user (Deleted)"] } }
                                    ]
                                })
     # Count the number of documents matching the filters
count = client.count(collection_name, query_filter)

In [43]:
count

CountResult(count=178)

In [24]:
1

1

In [55]:

# Function to check the number of documents that match a query and are above a certain threshold
def count_documents_above_threshold(collection_name, query_vector, threshold):
    # Search for documents matching the query vector and above the threshold
    results = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        search_params=models.SearchParams(
            exact = False, # m parameter for HNSW , number of edges per node 
            ),
        score_threshold=threshold
    )
    return len(results)

In [59]:
input_content_to_match = "we should use Guidewire for policy."
query_vector=embedding_model.encode(input_content_to_match).tolist()  # Example query vector


# 3. Count documents above a certain threshold
threshold = 0.5  # Example threshold
documents_above_threshold_count = count_documents_above_threshold(collection_name, query_vector, threshold)
print(f"Count of documents above threshold: {documents_above_threshold_count}")


Count of documents above threshold: 3


In [27]:
1

1

In [28]:
# Function to get top 5 matches for each type of algorithm in Qdrant
def get_top_5_matches_for_all_algorithms(collection_name, query_vector):
    # Define the types of distance algorithms available in Qdrant
    distance_algorithms = ["Cosine", "Dot", "Euclid", "Manhattan"]

    # Create a dictionary to store the top 5 results for each algorithm
    results = {}

    # Iterate over each distance algorithm
    for algorithm in distance_algorithms:
        # Search for the top 5 matches using the current algorithm
        search_params = SearchParams(
            hnsw_params=HnswParams(
                ef=200, # ef parameter for HNSW
                m=16    # m parameter for HNSW
            )
        )
        if algorithm == "Cosine":
            search_params.distance = "Cosine"
        elif algorithm == "Dot":
            search_params.distance = "Dot"
        elif algorithm == "Euclid":
            search_params.distance = "Euclid"
        elif algorithm == "Manhattan":
            search_params.distance = "Manhattan"

        top_matches = client.search(
            collection_name=collection_name,
            query_vector=query_vector,
            limit=5,
            search_params=search_params
        )

        # Store the top matches in the results dictionary
        results[algorithm] = top_matches

    return results

1

### Metadata key + content filtering 

In [65]:
# ## try this for content filtering 
# import hashlib
# import qdrant_client
# from qdrant_client.http.models import Filter, SearchParams, VectorParams, ScoredPoint

# # Initialize Qdrant client
# client = qdrant_client.QdrantClient(url="http://localhost:6333")

# # Function to convert page_id to a consistent integer
# def get_consistent_id(page_id):
#     return int(hashlib.md5(page_id.encode()).hexdigest(), 16) % (10 ** 8)


In [67]:

# Function to check the number of documents for a specific metadata filter and ensure the tag is not blank
def count_documents_with_tag(collection_name, tag):
    # Define a filter for the specified tag and non-blank content
    filter_criteria = models.Filter(
        must=[
            models.Match(key="tag", value=tag),
            models.Match(key="content", value={"$ne": ""})  # Ensure the content is not blank
        ]
    )

    # Search for documents matching the filter criteria
    results = client.scroll(
        collection_name=collection_name,
        filter=filter_criteria,
        limit=1000,  # Arbitrary limit, adjust as needed
        with_payload=True
    )

    # Count the number of matching documents
    count = len(results["points"])
    return count

# Example usage:
collection_name = "rnw_adr_collection_3"
tag = "problem"  # Example tag

documents_with_tag_count = count_documents_with_tag(collection_name, tag)
print(f"Count of documents with tag '{tag}' and non-blank content: {documents_with_tag_count}")

TypeError: Cannot instantiate typing.Union

In [71]:

# Function to check the number of documents for a specific metadata filter and ensure the tag is not blank
def count_documents_with_tag(collection_name, tag):
    # Define a filter for the specified tag and non-blank content
    filter_criteria =models.Filter(**{
                                    "must": [
                                              {
                                                "key": "tag",
                                                "match": {
                                                  "any": ["context"]
                                                }
                                              },
                                              {
                                                  "key" : "content",
                                                  "match" : {
                                                    "any" : [{"$ne":""}]
                                                  }
                                              }
                                            ]        
                                        })

    # Search for documents matching the filter criteria
    results = client.scroll(
        collection_name=collection_name,
        filter=filter_criteria,
        limit=1000,  # Arbitrary limit, adjust as needed
        with_payload=True
    )

    # Count the number of matching documents
    count = len(results["points"])
    return count

# Example usage:
collection_name = "rnw_adr_collection_3"
tag = "problem"  # Example tag

documents_with_tag_count = count_documents_with_tag(collection_name, tag)
print(f"Count of documents with tag '{tag}' and non-blank content: {documents_with_tag_count}")

ValidationError: 22 validation errors for Filter
must.1.FieldCondition.match.MatchValue.value
  Field required [type=missing, input_value={'any': [{'$ne': ''}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing
must.1.FieldCondition.match.MatchValue.any
  Extra inputs are not permitted [type=extra_forbidden, input_value=[{'$ne': ''}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.7/v/extra_forbidden
must.1.FieldCondition.match.MatchText.text
  Field required [type=missing, input_value={'any': [{'$ne': ''}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing
must.1.FieldCondition.match.MatchText.any
  Extra inputs are not permitted [type=extra_forbidden, input_value=[{'$ne': ''}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.7/v/extra_forbidden
must.1.FieldCondition.match.MatchAny.any.list[str].0
  Input should be a valid string [t

In [72]:
# store data again, in form of json/dict, to easily index/filter during search 

In [30]:
1

1